In [2]:
# AI used: Gemini
# Import pandas for data reading, manipulation, and time series handling
import pandas as pd
# Import Google Drive module for Colab environment mounting
from google.colab import drive

# Mount Google Drive to gain access to files in MyDrive
drive.mount('/content/drive')

# Store file path for FY19-FY22 encounters CSV
path1 = '/content/drive/MyDrive/ENSO_project/migration/sbo-encounters-fy19-fy22.csv'
# Store file path for FY23-FY26 encounters CSV
path2 = '/content/drive/MyDrive/ENSO_project/migration/sbo-encounters-fy23-fy26-aug.csv'

# Read first CSV file into a DataFrame
df1 = pd.read_csv(path1)
# Read second CSV file into a DataFrame
df2 = pd.read_csv(path2)

# Check whether column structures in both DataFrames match identically
columns_match = list(df1.columns) == list(df2.columns)
# Print confirmation of whether the dataset columns match
print(f"Columns match exactly: {columns_match}")

# Stack (concatenate) both DataFrames into a single combined DataFrame
df = pd.concat([df1, df2], ignore_index=True)

# Define list of categorical columns to inspect for unique values
cat_cols = ['Citizenship Grouping', 'Demographic', 'Component', 'Title of Authority', 'Encounter Type']
# Loop through each designated categorical column
for col in cat_cols:
    # Print sorted list of unique non-null values for the current column
    print(f"\nUnique values in '{col}':\n{sorted(df[col].dropna().unique())}")

# Print minimum and maximum Fiscal Year present in the first dataset
print(f"\nFY Range in File 1: {df1['Fiscal Year'].min()} to {df1['Fiscal Year'].max()}")
# Print minimum and maximum Fiscal Year present in the second dataset
print(f"FY Range in File 2: {df2['Fiscal Year'].min()} to {df2['Fiscal Year'].max()}")

# Extract set of unique (Fiscal Year, Month) pairs from the first dataset
fy_months_1 = set(zip(df1['Fiscal Year'], df1['Month (abbv)'].str.strip().str.upper()))
# Extract set of unique (Fiscal Year, Month) pairs from the second dataset
fy_months_2 = set(zip(df2['Fiscal Year'], df2['Month (abbv)'].str.strip().str.upper()))
# Intersect both sets to find any overlapping Fiscal Year and Month entries
overlap = fy_months_1.intersection(fy_months_2)

# Check if any overlapping pairs exist between the two datasets
if overlap:
    # Print overlapping Fiscal Year and Month combinations
    print(f"\nOverlapping Fiscal Year and Month pairs found ({len(overlap)}): {sorted(overlap)}")
# Handle case where no overlapping pairs are found
else:
    # Print confirmation that no months overlap across files
    print("\nNo overlapping Fiscal Year and Month pairs found between the two files.")

# Filter combined dataset to keep only rows where Citizenship Grouping is Honduras
df_honduras = df[df['Citizenship Grouping'].str.strip().str.title() == 'Honduras'].copy()

# Clean 'Fiscal Year' column by keeping only the digits (e.g., converting '2026 (FYTD)' to '2026')
df_honduras['Fiscal Year'] = df_honduras['Fiscal Year'].astype(str).str.extract(r'(\d{4})')[0].astype(int)

# Define mapping dictionary to convert month abbreviations into numeric values
month_map = {'OCT': 10, 'NOV': 11, 'DEC': 12, 'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8, 'SEP': 9}
# Create numerical month column from Month (abbv) strings
df_honduras['Month_Num'] = df_honduras['Month (abbv)'].str.strip().str.upper().map(month_map)

# Calculate Calendar Year: subtract 1 from FY for Oct-Dec, otherwise keep FY
df_honduras['Calendar_Year'] = df_honduras.apply(lambda row: int(row['Fiscal Year']) - 1 if row['Month_Num'] >= 10 else int(row['Fiscal Year']), axis=1)

# Construct first-of-the-month calendar date column from Calendar Year and Month Number
df_honduras['Date'] = pd.to_datetime(df_honduras['Calendar_Year'].astype(str) + '-' + df_honduras['Month_Num'].astype(str).str.zfill(2) + '-01')

# Group by calendar Date and calculate total monthly encounter count for Honduras
honduras_monthly = df_honduras.groupby('Date')['Encounter Count'].sum().reset_index().sort_values('Date')

# Display total monthly encounters header
print("\nTotal Honduran Encounters per Month:")
# Print the full table of aggregated monthly encounter totals
print(honduras_monthly.to_string(index=False))

# Identify first recorded month in the Honduran time series
first_month = honduras_monthly['Date'].min().strftime('%Y-%m')
# Identify last recorded month in the Honduran time series
last_month = honduras_monthly['Date'].max().strftime('%Y-%m')

# Print earliest recorded month
print(f"\nFirst Month: {first_month}")
# Print latest recorded month
print(f"Last Month: {last_month}")

# Generate complete month-by-month date range between min and max date
expected_dates = pd.date_range(start=honduras_monthly['Date'].min(), end=honduras_monthly['Date'].max(), freq='MS')
# Compute set difference to identify any missing dates in the sequence
missing_dates = expected_dates.difference(honduras_monthly['Date'])

# Check if any expected months are missing from the time series
if len(missing_dates) > 0:
    # Print list of missing months formatted as YYYY-MM
    print(f"\nMissing Months: {[d.strftime('%Y-%m') for d in missing_dates]}")
# Handle case where time series is continuous with zero missing months
else:
    # Print confirmation of continuous monthly sequence
    print("\nNo missing months in the time series.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Columns match exactly: True

Unique values in 'Citizenship Grouping':
['El Salvador', 'Guatemala', 'Honduras', 'Mexico', 'Other']

Unique values in 'Demographic':
['Accompanied Minors', 'FMUA', 'Single Adults', 'UAC', 'UC / Single Minors']

Unique values in 'Component':
['Office of Field Operations', 'U.S. Border Patrol']

Unique values in 'Title of Authority':
['Title 42', 'Title 8']

Unique values in 'Encounter Type':
['Apprehensions', 'Expulsions', 'Inadmissibles']

FY Range in File 1: 2019 to 2022
FY Range in File 2: 2023 to 2026 (FYTD)

No overlapping Fiscal Year and Month pairs found between the two files.

Total Honduran Encounters per Month:
      Date  Encounter Count
2018-10-01          13059.0
2018-11-01          14409.0
2018-12-01          14717.0
2019-01-01          13440.0
2019-02-01          22610.0
2019-03-01          29390.0
2019-04-01       